# Reference Resolution Dev Notebook

In [14]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER
import pandas as pd

from torch.optim import AdamW

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

## RefCOCO Data

### Utility Functions

### Import  Data

In [15]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=10.52s)


In [16]:
refer.IMAGE_DIR = '/home/claytonfields/nlp/code/data/coco/images/mscoco/train2014'

## METER Model

In [17]:
_config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])

# dm = MTDataModule(_config, dist=False)
model = METERTransformerSS(_config)
# exp_name = f'{_config["exp_name"]}'
# os.makedirs(_config["log_dir"], exist_ok=True)

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense.bias']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## To Do: Write New Data Class for Ref Res with multiple samples

1. Deliver single sentence with sub-images and text_labels, masks and ids so that METER can perform all subimage at once.

2.  May require padding to max_num_bb = 75

In [22]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, split='', max_bb = 40):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []
        

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            ref = self.refer.Refs[ref_id]
            for sent_id in ref['sent_ids']:
                sent_ids.append(sent_id)
        return sent_ids

    def __getitem__(self, index):
        sent_id = self.sent_ids[index]
        ref = self.refer.sentToRef[sent_id]
        sent = self.refer.Sents[sent_id]
        
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = self.refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        
        sub_images = []
        for obj in objs:
            try:
                x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            except ValueError:
                print(f'ValueError at setence id: {sent_id}')
                self.duds.append(sent_id)
                break
                
            if x_a is not None:
                sub_images.append(x_a)
        num_sub_images = len(sub_images)      
            
        text_ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        text_masks = [1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]
        text_labels = [[-100 for i in range(40)]]

#         ids = [text_ids for i in range(num_sub_images)]
#         masks = [text_masks for _ in range(num_sub_images)]
#         labels = [text_labels for i in range(num_sub_images)]
            
        return_dict = {
            'ann_id' : ann_id,
            'image' : sub_images,
            'obj_ids' : obj_ids,
            'sent_id' : sent_id,
            'text' : sent['sent'],
            'text_ids' : torch.tensor(text_ids),
            'text_labels' : torch.tensor(text_labels),
            'text_masks' : torch.tensor(text_masks)
        }  

        return return_dict

## Ref Res with METER

In [23]:
optimizer = AdamW(model.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 1

epochs = 1
# loader = dm.train_dataloader()
optim = AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.functional.cross_entropy
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [24]:
train_ds = RefcocoDataset(refer, tokenizer, split='train')
# train_ds = torch.utils.data.Subset(ds, sent_ids[729:])


In [21]:
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

training_loader = torch.utils.data.DataLoader(train_ds, **train_params)

In [9]:
eval_ds = RefcocoDataset(refer, tokenizer, split='val')
eval_ds.__len__()

10834

In [10]:
eval_params = {'batch_size': BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }
eval_loader = torch.utils.data.DataLoader(eval_ds, **eval_params)

In [11]:
# def train(model, training_ds, optimizer, loss_fn, device):
#     model.to(device)
#     model.train()
#     losses = []
#     for data in tqdm(training_ds):
#         if data['sent_id'] in training_ds.duds:
#             continue
#         try:
#             optimizer.zero_grad()
            
#             sent_id = data['sent_id']
#             infer_dict = model.infer(data)
#             logits = model.ref_classifier(infer_dict['cls_feats'])

#             obj_ids = data['obj_ids']
#             ann_id = data['ann_id']

#             target = torch.tensor([obj_ids.index(ann_id)])
#             loss = loss_fn(logits.reshape(1,-1),target)
#             losses.append(loss.item())
#             loss.backward()

#             optimizer.step()
#         except RuntimeError:
#             print(f'Runtime Error at sent_id = {sent_id}')
#             training_ds.duds.append(sent_id)
#     return losses, loss

# def evaluate(model, eval_ds):
#     gold = []
#     with torch.no_grad():
#         for data in tqdm(eval_ds):
#             if data['sent_id'] in training_ds.duds:
#                 continue
#             try:
#                 sent_id = data['sent_id']
#                 infer_dict = model.infer(data)
#                 logits = model.ref_classifier(infer_dict['cls_feats'])

#                 obj_ids = data['obj_ids']
#                 ann_id = data['ann_id']

#                 pred_index = np.argmax(logits)
#                 pred_id = obj_ids[pred_index]
#                 target = torch.tensor([obj_ids.index(ann_id)])
#                 if pred_id == ann_id:
#                     gold.append(1)
#                 else:
#                     gold.append(0)
#             except RuntimeError:
#                 print(f'RuntimeError at sent_id = {sent_id}')
#                 eval_ds.duds.append(sent_id)
#     return gold

In [12]:
# for epoch in range(epochs):
#     losses, loss = train(model, train_ds, optimizer, loss_fn, device)
#     print(f'Epoch: {epoch}, Loss:  {loss.item()}')  
#     loss_frame = pd.DataFrame(losses,columns=['Loss'])
#     gold = evaluate(model, eval_ds)
#     acc = np.average(gold)
#     print(f'acurracy on test set {acc}')

In [27]:
# model.to(device)
model.train()
losses = []
for data in tqdm(train_ds):

    optimizer.zero_grad()

    sent_id = data['sent_id']
    cls = []
    for i,sub_image in enumerate(data['image']):

        ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.
        input_dict = {
    #                 'image' : [sub_image.squeeze(dim=0)],
            'image' : [sub_image],
            'text' : data['text'],
            'text_ids' : data['text_ids'].reshape(1,-1),
            'text_labels' : data['text_labels'].reshape(1,-1),
            'text_masks' : data['text_masks'].reshape(1,-1)
        }
        infer_dict = model.infer(input_dict)

        cls.append(infer_dict['cls_feats'])
    
    cls_tensor = torch.cat(cls)
    logits = model.ref_classifier(cls_tensor)

    obj_ids = data['obj_ids']
    ann_id = data['ann_id']

    target = torch.tensor([obj_ids.index(ann_id)])
    loss = loss_fn(logits.reshape(1,-1),target)
    losses.append(loss.item())
    loss.backward()

    optimizer.step()


  0%|                                                | 0/120624 [00:02<?, ?it/s]


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [48]:
cls = []
for i,sub_image in enumerate(data['image']):

    ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.
    input_dict = {
#                 'image' : [sub_image.squeeze(dim=0)],
        'image' : [sub_image],
        'text' : data['text'],
        'text_ids' : data['text_ids'][i].reshape(1,-1),
        'text_labels' : data['text_labels'][i].reshape(1,-1),
        'text_masks' : data['text_masks'][i].reshape(1,-1)
    }
    infer_dict = model.infer(input_dict)
    
    cls.append(infer_dict['cls_feats'])
cls.__len__()

33

In [49]:
data['image'][0].shape

torch.Size([1, 3, 224, 224])

In [50]:
foo = torch.cat(cls)

In [23]:
foo[:, 0].shape

torch.Size([33])

In [63]:
bar = model.ref_classifier(foo)
bar

tensor([[0.2324],
        [0.1645],
        [0.1886],
        [0.3033],
        [0.6057],
        [0.7868],
        [0.5119],
        [0.7701],
        [0.3497],
        [0.6667],
        [0.8065],
        [0.5901],
        [0.6091],
        [0.4992],
        [0.6186],
        [0.5868],
        [0.2207],
        [0.5904],
        [0.8100],
        [0.6154],
        [1.0619],
        [0.7046],
        [0.4959],
        [0.4271],
        [0.6321],
        [0.3808],
        [0.4659],
        [0.6696],
        [0.5242],
        [0.5054],
        [0.5838],
        [0.9272],
        [0.4803]], grad_fn=<AddmmBackward>)

In [ ]:
torch.argmax(bar)

In [65]:
temp_input = data.copy()
temp_input['image'] = [torch.cat(data['image'])]

temp1 = model.infer(temp_input)
temp2 = model.ref_classifier(temp1['cls_feats'])

In [55]:
temp2

tensor([[0.2654],
        [0.3004],
        [0.2787],
        [0.4242],
        [0.5139],
        [0.7725],
        [0.6006],
        [0.9785],
        [0.3439],
        [0.6043],
        [0.7958],
        [0.9494],
        [0.6001],
        [0.5253],
        [0.4707],
        [0.4767],
        [0.3288],
        [0.5211],
        [0.8549],
        [0.6701],
        [0.8236],
        [0.5612],
        [0.7100],
        [0.7465],
        [0.6149],
        [0.4455],
        [0.4718],
        [0.6328],
        [0.4382],
        [0.4787],
        [0.5169],
        [0.9031],
        [0.4643]], grad_fn=<AddmmBackward>)

In [56]:
torch.argmax(temp2)

tensor(7)

In [57]:
obj_ids = data['obj_ids']
ann_id = data['ann_id']

target = torch.tensor([obj_ids.index(ann_id)])

In [58]:
target

tensor([16])

In [59]:
ann_id

1719310

In [60]:
obj_ids

[463958,
 467023,
 470153,
 1050612,
 1050827,
 1051359,
 1542292,
 1543882,
 1544192,
 1544712,
 1545028,
 1545441,
 1546195,
 1546665,
 1558299,
 1558781,
 1719310,
 1904706,
 1904982,
 1905032,
 1905335,
 1906430,
 1907189,
 1907666,
 1907898,
 1908215,
 1908291,
 1908329,
 2077654,
 2077803,
 2078079,
 2110251,
 905200581857]